> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 12 · Capstone: make CampusMate yours

⏱️ **60 minutes** (5 min brief + 45 min build + 10 min demos)

*All data is fictional.*

## The brief

Add **one new tool** and **one new Cedar policy** of your choice to CampusMate, run it, and show the trace.
Policies only see Gateway tools, so your tool goes on the Gateway as a Lambda (same steps as Labs 7 and 11).

### Five ideas
| # | Tool | Policy idea |
|---|---|---|
| 1 | `register_for_event(event_id, student_id, team_size)` over `data/events.json` | forbid when `team_size > 4` |
| 2 | `library_due_date(book_id, days)` | forbid renewals longer than 14 days |
| 3 | `request_fee_extension(student_id, days)` | forbid extensions above 7 days |
| 4 | `book_lab_slot(lab, hours)` | forbid bookings longer than 3 hours |
| 5 | `apply_od(student_id, event_id, days)` for hackathons | forbid OD above 2 days |

The template below does **idea 1** end to end, with `# TODO`s where you make it your own.

## Grading rubric (20 points)

| Criterion | 0 | 2 | 4 |
|---|---|---|---|
| **Tool works** | missing | runs, rough edges | clear schema + docstring, handles bad input |
| **Agent uses it** | never called | called sometimes | called reliably for the right questions |
| **Cedar policy** | missing / fails | deployed | deployed, minimal, and you can explain every line |
| **Policy proven** | not tested | allowed case shown | allowed **and** denied cases shown |
| **Trace + explanation** | none | trace shown | trace shown + you explain latency/tokens/tool choice |

Bonus +2: a rule about totals (needs state), or an evaluator idea that would catch a bad answer for your tool.

## Settings

✏️ Change `STUDENT_ID` to your own ID before running anything else.

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib, subprocess, shutil
import boto3

# ✏️ Put YOUR student ID here (the same one in every notebook). Lower-case letters and digits, e.g. priya07
STUDENT_ID = "student01"

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"

PROJECT = f"campusmate{STUDENT_ID}"                    # your AgentCore project (created in notebook 05)
PROJECT_DIR = ROOT / PROJECT                           # the folder it lives in
AGENT_FILE = PROJECT_DIR / "app" / "agent" / "main.py" # your agent's code

def agentcore(*args, folder=None):
    """Run an AgentCore CLI command (the same one you would type in a terminal) and show its output."""
    process = subprocess.Popen([shutil.which("agentcore") or "agentcore", *args], cwd=folder or PROJECT_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "[SDK info]" not in line:
            print(line, end="")
    if process.wait() != 0:
        raise RuntimeError("The agentcore command failed. Read the messages above.")

print("Student:", STUDENT_ID, "| Project folder:", PROJECT_DIR)

## What you built in earlier labs

The AgentCore CLI knows everything in your project. We ask it, so there are no IDs to copy.

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## Step 1: write the Lambda tool (idea 1 template)

In [ ]:
CAPSTONE_LAMBDA = f"campusmate-capstone-{STUDENT_ID}"
LAMBDA_CODE = """
import json
EVENTS = json.load(open("events.json"))["events"]

def lambda_handler(event, context):
    ev = next((e for e in EVENTS if e["event_id"] == str(event.get("event_id", "")).upper()), None)
    if not ev:
        return {"registered": False, "message": "Unknown event id"}
    seats_left = ev["registration_cap"] - ev["registered"]
    # TODO: decide what else to return (venue, date, a reference number...)
    return {"registered": seats_left > 0, "event": ev["title"], "date": ev["date"],
            "team_size": event.get("team_size"), "seats_left_before": seats_left}
"""
TOOL_SCHEMA = [{
    "name": "register_for_event",
    "description": "Register a student team for a campus event (hackathon, workshop, club meet) by event ID such as EVT-03.",  # TODO: improve
    "inputSchema": {"type": "object", "required": ["event_id", "student_id", "team_size"], "properties": {
        "event_id": {"type": "string", "description": "Event ID like EVT-03"},
        "student_id": {"type": "string", "description": "Student ID like AIDD-DEMO-001"},
        "team_size": {"type": "integer", "description": "Number of people in the team"}}}}]

capstone_dir = PROJECT_DIR / "capstone"
capstone_dir.mkdir(exist_ok=True)
(capstone_dir / "handler.py").write_text(LAMBDA_CODE)
(capstone_dir / "tool_schema.json").write_text(json.dumps(TOOL_SCHEMA, indent=2))
print("Tool will be called: events___register_for_event")

## Step 2: create the Lambda and add it to your Gateway

In [ ]:
iam = boto3.client("iam")
lam = boto3.client("lambda", region_name=REGION)
LAMBDA_ROLE = f"campusmate-lambda-role-{STUDENT_ID}"

# Every Lambda function needs a role. This one only lets it write its own logs (an AWS-managed policy).
try:
    lambda_role_arn = iam.get_role(RoleName=LAMBDA_ROLE)["Role"]["Arn"]
except iam.exceptions.NoSuchEntityException:
    trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"},
                                                     "Action": "sts:AssumeRole"}]}
    lambda_role_arn = iam.create_role(RoleName=LAMBDA_ROLE, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
    iam.attach_role_policy(RoleName=LAMBDA_ROLE, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
    time.sleep(10)   # a new role takes a few seconds to be ready

def create_lambda(name, handler_file, extra_files=()):
    """Zip the handler (plus any data files) and create or update the Lambda function."""
    import io, zipfile
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w") as z:
        z.write(handler_file, "handler.py")
        for path in extra_files:
            z.write(path, pathlib.Path(path).name)
    try:
        lam.update_function_code(FunctionName=name, ZipFile=buf.getvalue())
        lam.get_waiter("function_updated_v2").wait(FunctionName=name)
    except lam.exceptions.ResourceNotFoundException:
        lam.create_function(FunctionName=name, Runtime="python3.12", Role=lambda_role_arn, Handler="handler.lambda_handler",
                            Code={"ZipFile": buf.getvalue()}, Architectures=["arm64"], Timeout=15)
        lam.get_waiter("function_active_v2").wait(FunctionName=name)
    return lam.get_function(FunctionName=name)["Configuration"]["FunctionArn"]

capstone_arn = create_lambda(CAPSTONE_LAMBDA, capstone_dir / "handler.py", [DATA_DIR / "events.json"])
if '"events"' not in (PROJECT_DIR / "agentcore" / "agentcore.json").read_text():
    agentcore("add", "gateway-target", "--gateway", "toolsGateway", "--target-configuration", json.dumps({
        "name": "events", "targetType": "lambdaFunctionArn",
        "lambdaFunctionArn": {"lambdaArn": capstone_arn, "toolSchemaFile": str(capstone_dir / "tool_schema.json")}}))
agentcore("deploy")

## Step 3: your Cedar policy

The allow policy from Lab 11 already permits every tool on your Gateway for callers in our account, so you only need a `forbid`.

In [ ]:
GATEWAY_ARN = deployed()["toolsGateway"]["arn"]
MY_CEDAR = f"""forbid(
  principal is AgentCore::IamEntity,
  action == AgentCore::Action::"events___register_for_event",
  resource == AgentCore::Gateway::"{GATEWAY_ARN}"
)
when {{ context.input.team_size > 4 }};"""   # TODO: change the condition for your idea

if "capstone_rule" not in (PROJECT_DIR / "agentcore" / "agentcore.json").read_text():
    agentcore("add", "policy", "--engine", "campusRules", "--name", "capstone_rule", "--statement", MY_CEDAR,
              "--validation-mode", "ignore-all-findings")
agentcore("deploy")

## Step 4: prove it, allowed and denied

In [ ]:
from mcp import ClientSession
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

GATEWAY_URL = boto3.client("bedrock-agentcore-control", region_name=REGION).get_gateway(
    gatewayIdentifier=deployed()["toolsGateway"]["arn"].split("/")[-1])["gatewayUrl"] + "/mcp"

async def call_tool(tool=None, args=None):
    """Talk MCP to your Gateway: list its tools, or call one. Signed with YOUR AWS identity."""
    async with aws_iam_streamablehttp_client(endpoint=GATEWAY_URL, aws_service="bedrock-agentcore", aws_region=REGION) as (read, write, _):
        async with ClientSession(read, write) as s:
            await s.initialize()
            if tool is None:
                return [t.name for t in (await s.list_tools()).tools]
            try:
                r = await s.call_tool(tool, args)
                return ("DENIED: " if r.isError else "ALLOWED: ") + r.content[0].text[:200]
            except Exception as e:
                return f"DENIED: {e}"

time.sleep(15)
print("team of 3:", await call_tool("events___register_for_event", {"event_id": "EVT-03", "student_id": "AIDD-DEMO-001", "team_size": 3}))
print("team of 6:", await call_tool("events___register_for_event", {"event_id": "EVT-03", "student_id": "AIDD-DEMO-001", "team_size": 6}))

## Step 5: let your agent use it, then show the trace

The agent finds Gateway tools when a conversation starts, so a **new session** sees your tool.

In [ ]:
session = f"capstone-{uuid.uuid4()}"
print(ask("Register my team of 3 for the HackAIDD hackathon (EVT-03).", session))
print("\nTrace: CloudWatch -> GenAI Observability -> Bedrock AgentCore -> your agent -> Sessions ->", session)

## ✍️ Your turn

Make it yours: change the idea, the tool code, the description and the Cedar condition. Prepare a 2-minute demo: allowed case, denied case, and the trace.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
DEMO_NOTES = {"my_idea": "TODO", "allowed_example": "TODO", "denied_example": "TODO", "what_the_trace_shows": "TODO"}
print(json.dumps(DEMO_NOTES, indent=1))

## ✅ Checkpoint

You are on track if:
- Your target and policy deployed
- One call ALLOWED and one DENIED by your policy
- Your agent called your tool in a new session

**Stuck or behind?** Copy from `this notebook's template, and notebooks 07 + 11` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| Deploy fails: *unable to find an applicable action* | The policy was added before the tool was deployed | Run Step 2's deploy first, then Step 3 |
| Agent doesn't see the new tool | An old session cached its tool list | Use a NEW session ID |
| Every call to your tool DENIED | The Lab 11 allow policy is missing | Finish notebook 11 first |